## GLOBAL — Reproducibility + Colab GPU Budget Setup

In [ ]:
import torch, random, numpy as np, gc, os

torch.manual_seed(42); random.seed(42); np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# ── Colab GPU Budget Settings ──────────────────────────────────────────────
# Everything is controlled here. Tune these if you have more/less GPU time.
#
# Default values are designed so the ENTIRE notebook (training + ablations
# + evaluation) finishes in ~60–75 min on Colab T4 (free tier).
#
TRAIN_STEPS   = 20     # steps per epoch  (20 steps × 2 epochs = 40 updates)
EVAL_STEPS    = 15     # questions to evaluate per model
N_SAMPLES     = 2      # outputs per question during training (was 4 → 2× faster)
EVAL_RUNS     = 4      # inference runs per question during eval (was 8 → 4× faster)
MAX_NEW_TOK   = 120    # max tokens to generate (shorter = faster)
EPOCHS        = 2      # training epochs
ABLATION_STEPS= 15     # steps per ablation run (less than main training)

print(f"GPU  : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM : {vram:.1f} GB")
print(f"Config → train_steps={TRAIN_STEPS}, eval_steps={EVAL_STEPS}, "
      f"n_samples={N_SAMPLES}, eval_runs={EVAL_RUNS}, "
      f"max_new_tokens={MAX_NEW_TOK}, epochs={EPOCHS}")

GPU  : Tesla T4
VRAM : 15.6 GB
Config → train_steps=20, eval_steps=15, n_samples=2, eval_runs=4, max_new_tokens=120, epochs=2


## STAGE 1 — Install + Load Model

In [ ]:
!pip install -q transformers datasets accelerate peft bitsandbytes sympy

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 4-bit quantization — keeps model under 4 GB VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config,
    trust_remote_code=True
)
model.config.use_cache = False
print("Model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded.


In [ ]:
# Quick smoke-test
dev = "cuda" if torch.cuda.is_available() else "cpu"
inp = tokenizer("What is 6 x 7? Show steps as: A op B = C",
                return_tensors="pt").to(dev)
with torch.no_grad():
    out = model.generate(**inp, max_new_tokens=60,
                         pad_token_id=tokenizer.pad_token_id)
print(tokenizer.decode(out[0], skip_special_tokens=True))

What is 6 x 7? Show steps as: A op B = C, where A=6, B=7 and C=? To solve the problem \( 6 \times 7 \), follow these steps:

1. **Identify the numbers**: The given numbers are \( 6 \) and \( 7 \).

2. **Multiply the numbers**: Perform


## STAGE 2 — Dataset + No-RL Baseline

In [ ]:
from datasets import load_dataset

dataset  = load_dataset("gsm8k", "main")
# Load enough data for training + eval; we index into it per stage
all_data = dataset["train"].select(range(60))   # 60 total: 20 train, rest for eval
eval_pool= dataset["test"].select(range(EVAL_STEPS))

print(f"Training pool : 60 samples (use first {TRAIN_STEPS} per epoch)")
print(f"Eval pool     : {len(eval_pool)} samples")
print(all_data[0])

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Training pool : 60 samples (use first 20 per epoch)
Eval pool     : 15 samples
{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}


In [ ]:
import re

def extract_answer(text):
    """Extract final numeric answer from model output or GSM8K label."""
    if "####" in text:
        try:
            return float(text.split("####")[-1].strip().replace(",",""))
        except:
            pass
    # 'answer is X' or '= X' at end
    m = re.search(r"(?:answer is|the answer is|=)\s*\$?([\-]?[\d,]+\.?\d*)", text, re.I)
    if m:
        try:
            return float(m.group(1).replace(",",""))
        except:
            pass
    nums = re.findall(r"[\-]?\d+\.?\d*", text)
    return float(nums[-1]) if nums else None

In [ ]:
def make_prompt(question):
    """Adds formatting hint so R2 parser has equations to check."""
    return f"{question}\nWrite each calculation as: number op number = result"

def gen_one(question, mdl=None):
    """Single-output generation (used for baseline eval)."""
    if mdl is None: mdl = model
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    inp = tokenizer(make_prompt(question), return_tensors="pt").to(dev)
    with torch.no_grad():
        out = mdl.generate(
            **inp, max_new_tokens=MAX_NEW_TOK,
            do_sample=True, temperature=0.7,
            pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0], skip_special_tokens=True)

In [ ]:
# No-RL baseline — only on EVAL_STEPS questions to save time
correct = 0
for i in range(EVAL_STEPS):
    s  = eval_pool[i]
    gt = extract_answer(s["answer"])
    p  = extract_answer(gen_one(s["question"]))
    if p is not None and gt is not None and abs(p - gt) < 1e-4:
        correct += 1
    if (i+1) % 5 == 0:
        print(f"  {i+1}/{EVAL_STEPS}")

baseline_no_rl_acc = correct / EVAL_STEPS
print(f"\nNo-RL Baseline Accuracy: {baseline_no_rl_acc:.3f}")

  5/15
  10/15
  15/15

No-RL Baseline Accuracy: 0.000


## STAGE 3 — Reward System

### 3.1 Multi-Sample Generation

In [ ]:
def gen_multiple(question, mdl=None, n=None):
    """
    Generate N outputs. Uses num_return_sequences (single forward pass).
    Falls back to sequential if OOM — safe on all Colab GPUs.
    """
    if mdl is None: mdl = model
    if n   is None: n   = N_SAMPLES
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    inp = tokenizer(make_prompt(question), return_tensors="pt").to(dev)
    try:
        with torch.no_grad():
            outs = mdl.generate(
                **inp, max_new_tokens=MAX_NEW_TOK,
                do_sample=True, temperature=0.7,
                num_return_sequences=n,
                pad_token_id=tokenizer.pad_token_id)
        return [tokenizer.decode(o, skip_special_tokens=True) for o in outs]
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); gc.collect()
        results = []
        for _ in range(n):
            with torch.no_grad():
                o = mdl.generate(
                    **inp, max_new_tokens=MAX_NEW_TOK,
                    do_sample=True, temperature=0.7,
                    pad_token_id=tokenizer.pad_token_id)
            results.append(tokenizer.decode(o[0], skip_special_tokens=True))
        return results

### 3.2 Reward Functions

In [ ]:
# ── R1 : Correctness ──────────────────────────────────────────────────────
def reward_r1(pred, gt):
    if pred is None or gt is None: return 0.0
    return 1.0 if abs(pred - gt) < 1e-4 else 0.0

# ── R2 : Step Validity (rule-based arithmetic) ────────────────────────────
def reward_r2(text):
    """
    Parse lines matching  A op B = C  and verify with Python arithmetic.
    Skip-not-penalize: unparsable lines are ignored, not scored 0.
    """
    pat = r"([\-]?\d+\.?\d*)\s*([+\-*/x×])\s*([\-]?\d+\.?\d*)\s*=\s*([\-]?\d+\.?\d*)"
    valid = total = 0
    for line in text.split("\n"):
        m = re.search(pat, line)
        if not m: continue
        total += 1
        try:
            a, op, b, c = m.groups()
            a, b, c = float(a), float(b), float(c)
            res = (a+b if op=="+" else a-b if op=="-"
                   else a*b if op in ("*","x","×")
                   else (a/b if b!=0 else None) if op=="/" else None)
            if res is not None and abs(res - c) < 1e-4:
                valid += 1
        except:
            pass
    return 0.0 if total == 0 else valid / total

# ── R3 : Self-Consistency (GATED) ─────────────────────────────────────────
from collections import Counter

def reward_r3(predictions, gt):
    """
    Agreement rate across N outputs.
    GATE: returns 0 if majority answer is wrong → no reward for stable-wrong.
    """
    answers = [extract_answer(p) for p in predictions]
    answers = [a for a in answers if a is not None]
    if not answers: return 0.0
    majority, freq = Counter(answers).most_common(1)[0]
    if gt is not None and abs(majority - gt) < 1e-4:
        return freq / len(predictions)
    return 0.0

# ── R4 : Verbosity Penalty ────────────────────────────────────────────────
_lens = sorted(len(s["answer"].split()) for s in all_data)
VERBOSITY_THRESHOLD = 1.5 * _lens[len(_lens)//2]
print(f"Verbosity threshold: {VERBOSITY_THRESHOLD:.0f} words")

def reward_r4(text):
    l = len(text.split())
    if l <= VERBOSITY_THRESHOLD: return 0.0
    return min(1.0, (l - VERBOSITY_THRESHOLD) / VERBOSITY_THRESHOLD)

# ── Combined reward ───────────────────────────────────────────────────────
def compute_reward(R1, R2, R3, R4):
    return 0.5*R1 + 0.3*R2 + 0.15*R3 - 0.05*R4

Verbosity threshold: 80 words


In [ ]:
# ── Quick reward sanity check ─────────────────────────────────────────────
for idx, label in [(0,"easy"),(10,"hard")]:
    s   = all_data[idx]
    gt  = extract_answer(s["answer"])
    outs= gen_multiple(s["question"])
    p   = extract_answer(outs[0])
    r1  = reward_r1(p, gt)
    r2  = reward_r2(outs[0])
    r3  = reward_r3(outs, gt)
    r4  = reward_r4(outs[0])
    print(f"[{label}] GT={gt} Pred={p} | R1={r1:.2f} R2={r2:.2f} "
          f"R3={r3:.2f} R4={r4:.2f} → Total={compute_reward(r1,r2,r3,r4):.3f}")

[easy] GT=72.0 Pred=24.0 | R1=0.00 R2=1.00 R3=0.00 R4=0.36 → Total=0.282
[hard] GT=121.0 Pred=3.0 | R1=0.00 R2=0.00 R3=0.00 R4=0.99 → Total=-0.049


## STAGE 4 — RL Training

### 4.1 LoRA Setup

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch.nn.functional as F

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


### 4.2 Generate with Log-Probability

In [ ]:
def gen_with_logprob(question, mdl):
    """
    Generate one response and return (text, log_prob_of_response).
    Prompt tokens are masked out — only response tokens contribute to logprob.
    """
    dev   = "cuda" if torch.cuda.is_available() else "cpu"
    prompt= make_prompt(question)
    inp   = tokenizer(prompt, return_tensors="pt").to(dev)
    plen  = inp.input_ids.shape[1]

    # Step 1: generate (no grad — saves memory)
    with torch.no_grad():
        gen = mdl.generate(
            **inp, max_new_tokens=MAX_NEW_TOK,
            do_sample=True, temperature=0.7,
            pad_token_id=tokenizer.pad_token_id)

    text = tokenizer.decode(gen[0], skip_special_tokens=True)

    # Step 2: re-tokenize full text and recompute logprobs WITH grad
    full = tokenizer(text, return_tensors="pt").to(dev)
    flen = full.input_ids.shape[1]

    # Safety: if response is empty after re-tokenization return zero grad tensor
    if flen <= plen:
        return text, torch.tensor(0.0, requires_grad=True, device=dev)

    with torch.enable_grad():
        out = mdl(**full)

    shift_logits = out.logits[:, :-1, :]
    shift_labels = full.input_ids[:, 1:]
    lp = F.log_softmax(shift_logits, dim=-1)
    sel= lp.gather(2, shift_labels.unsqueeze(-1)).squeeze(-1)

    # Response tokens only (mask prompt)
    resp_lp = sel[:, max(0, plen-1):]
    return text, resp_lp.sum()

### 4.3 Training Loop

In [ ]:
# Prepare training slice
train_qs  = list(all_data["question"])[:TRAIN_STEPS]
train_ans = list(all_data["answer"])[:TRAIN_STEPS]
print(f"Training on {len(train_qs)} questions × {EPOCHS} epochs = "
      f"{len(train_qs)*EPOCHS} total gradient steps")

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
model.train()

for epoch in range(EPOCHS):
    print(f"\n{'='*45}  Epoch {epoch+1}/{EPOCHS}  {'='*45}")
    log = {"R1":[],"R2":[],"R3":[],"R4":[],"raw":[],"norm":[]}

    for i, (q, ans) in enumerate(zip(train_qs, train_ans)):
        gt = extract_answer(ans)

        # Generate N_SAMPLES outputs
        texts, lps = [], []
        for _ in range(N_SAMPLES):
            t, lp = gen_with_logprob(q, model)
            texts.append(t); lps.append(lp)

        # Compute reward on first output (logprob from first sample)
        pred = extract_answer(texts[0])
        R1   = reward_r1(pred, gt)
        R2   = reward_r2(texts[0])
        R3   = reward_r3(texts, gt)
        R4   = reward_r4(texts[0])
        raw  = compute_reward(R1, R2, R3, R4)

        # Reward logging
        log["R1"].append(R1); log["R2"].append(R2)
        log["R3"].append(R3); log["R4"].append(R4)
        log["raw"].append(float(raw))

        # Batch-level normalisation (running stats within epoch)
        rv = log["raw"]
        if len(rv) >= 2:
            mu  = sum(rv)/len(rv)
            sig = (sum((r-mu)**2 for r in rv)/len(rv))**0.5
            rn  = (float(raw)-mu)/(sig+1e-8)
        else:
            rn  = float(raw)
        log["norm"].append(rn)

        loss = -rn * lps[0]

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        # Free memory every step — critical on Colab free tier
        del texts, lps, loss
        torch.cuda.empty_cache(); gc.collect()

        print(f"  step {i+1:>2}/{len(train_qs)} | "
              f"R1={R1:.2f} R2={R2:.2f} R3={R3:.2f} R4={R4:.2f} | "
              f"raw={float(raw):.3f} norm={rn:.3f}")

    def _m(l): return round(sum(l)/len(l),4) if l else 0
    print(f"\n  Epoch {epoch+1} avg → "
          f"R1={_m(log['R1'])} R2={_m(log['R2'])} "
          f"R3={_m(log['R3'])} R4={_m(log['R4'])} "
          f"raw={_m(log['raw'])} norm={_m(log['norm'])}")

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Training on 20 questions × 2 epochs = 40 total gradient steps

=============================================  Epoch 1/2  =============================================


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


  step  1/20 | R1=0.00 R2=0.00 R3=0.00 R4=0.00 | raw=0.000 norm=0.000
  step  2/20 | R1=0.00 R2=0.00 R3=0.00 R4=0.06 | raw=-0.003 norm=-1.000
  step  3/20 | R1=0.00 R2=0.00 R3=0.00 R4=0.38 | raw=-0.019 norm=-1.401
  step  4/20 | R1=0.00 R2=0.00 R3=0.00 R4=0.19 | raw=-0.010 norm=-0.244
  step  5/20 | R1=0.00 R2=0.00 R3=0.00 R4=0.13 | raw=-0.007 norm=0.161
  step  6/20 | R1=0.00 R2=0.00 R3=0.00 R4=0.18 | raw=-0.009 norm=-0.198
  step  7/20 | R1=0.00 R2=0.00 R3=0.00 R4=0.40 | raw=-0.020 norm=-1.456
  step  8/20 | R1=0.00 R2=0.00 R3=0.00 R4=1.00 | raw=-0.050 norm=-2.375
  step  9/20 | R1=0.00 R2=0.00 R3=0.00 R4=0.70 | raw=-0.035 norm=-1.168
  step 10/20 | R1=0.00 R2=0.00 R3=0.00 R4=0.17 | raw=-0.008 norm=0.511
  step 11/20 | R1=0.00 R2=0.00 R3=0.00 R4=0.27 | raw=-0.014 norm=0.164
  step 12/20 | R1=0.00 R2=0.00 R3=0.00 R4=0.43 | raw=-0.022 norm=-0.394
  step 13/20 | R1=0.00 R2=0.00 R3=0.00 R4=0.01 | raw=-0.000 norm=1.072
  step 14/20 | R1=0.00 R2=0.00 R3=0.00 R4=0.31 | raw=-0.015 norm=-0.02

## STAGE 5 — Evaluation

### 5.1 Load Baseline Model

In [ ]:
from transformers import AutoModelForCausalLM

# Load a fresh copy of the base model as baseline (identical quantization)
baseline_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config,
    trust_remote_code=True
)
baseline_model.config.use_cache = False
print("Baseline model loaded.")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Baseline model loaded.


### 5.2 Evaluation Function

In [ ]:
def evaluate_model(mdl, eval_dataset, num_q=None, label="Model"):
    """
    Evaluate accuracy, consistency, and step score on num_q questions.
    Uses EVAL_RUNS inference samples per question.
    Consistency threshold: majority in ≥ ceil(EVAL_RUNS * 0.6) outputs.
    """
    if num_q is None: num_q = EVAL_STEPS
    mdl.eval()

    correct = consistency_count = 0
    step_scores = []
    agree_thresh = max(2, round(EVAL_RUNS * 0.6))

    for i in range(num_q):
        s  = eval_dataset[i]
        q  = s["question"]
        gt = extract_answer(s["answer"])

        outs = []
        for _ in range(EVAL_RUNS):
            try:
                outs.append(gen_one(q, mdl))
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache(); gc.collect()
                break
        if not outs: continue

        # Accuracy
        p = extract_answer(outs[0])
        if p is not None and gt is not None and abs(p-gt) < 1e-4:
            correct += 1

        # Consistency (≥60% agreement)
        answers = [extract_answer(o) for o in outs]
        answers = [a for a in answers if a is not None]
        if answers:
            _, freq = Counter(answers).most_common(1)[0]
            if freq >= agree_thresh:
                consistency_count += 1

        # Step score (R2 on all outputs)
        r2s = [reward_r2(o) for o in outs]
        step_scores.append(sum(r2s)/len(r2s))

        if (i+1) % 5 == 0:
            print(f"  [{label}] {i+1}/{num_q}")

    acc   = correct / num_q
    cons  = consistency_count / num_q
    step  = sum(step_scores)/len(step_scores) if step_scores else 0.0
    return acc, cons, step

### 5.3 Run Evaluation

In [ ]:
print("── Baseline ──")
baseline_acc, baseline_cons, baseline_step = evaluate_model(
    baseline_model, eval_pool, label="Baseline")

print("\n── Trained RL Model ──")
model.eval()
trained_acc, trained_cons, trained_step = evaluate_model(
    model, eval_pool, label="Trained")

print("\n── COMPARISON ──────────────────────")
print(f"Baseline   → Acc:{baseline_acc:.3f}  Cons:{baseline_cons:.3f}  Step:{baseline_step:.3f}")
print(f"Trained RL → Acc:{trained_acc:.3f}  Cons:{trained_cons:.3f}  Step:{trained_step:.3f}")

── Baseline ──
  [Baseline] 5/15
  [Baseline] 10/15
  [Baseline] 15/15

── Trained RL Model ──
  [Trained] 5/15
  [Trained] 10/15
  [Trained] 15/15

── COMPARISON ──────────────────────
Baseline   → Acc:0.000  Cons:0.667  Step:0.186
Trained RL → Acc:0.000  Cons:0.733  Step:0.212


## STAGE 6 — Ablation Experiments + Final Results

### 6.1 Ablation Reward Functions

In [ ]:
# Weights renormalized so scale is consistent across ablations

def reward_full(R1,R2,R3,R4):   return 0.500*R1 + 0.300*R2 + 0.150*R3 - 0.050*R4
def reward_no_r2(R1,R2,R3,R4): return 0.769*R1 + 0.000*R2 + 0.231*R3 - 0.077*R4
def reward_no_r3(R1,R2,R3,R4): return 0.625*R1 + 0.375*R2 + 0.000*R3 - 0.063*R4
def reward_no_r4(R1,R2,R3,R4): return 0.500*R1 + 0.300*R2 + 0.150*R3 + 0.000*R4

### 6.2 Ablation Training Wrapper

In [ ]:
from peft import get_peft_model, prepare_model_for_kbit_training

def train_ablation(reward_fn, name):
    """Train a fresh model with a given reward function. Uses ABLATION_STEPS steps."""
    print(f"\n{'='*40}  {name}  {'='*40}")

    # Fresh quantized model every time
    tmp = AutoModelForCausalLM.from_pretrained(
        model_name, device_map="auto",
        quantization_config=bnb_config, trust_remote_code=True)
    tmp.config.use_cache = False
    tmp = prepare_model_for_kbit_training(tmp)
    tmp = get_peft_model(tmp, lora_config)
    tmp.train()

    opt = torch.optim.AdamW(tmp.parameters(), lr=1e-5)

    # Use first ABLATION_STEPS questions (subset of training data)
    abl_qs  = train_qs[:ABLATION_STEPS]
    abl_ans = train_ans[:ABLATION_STEPS]

    for epoch in range(EPOCHS):
        rewards_ep = []
        for i, (q, ans) in enumerate(zip(abl_qs, abl_ans)):
            gt   = extract_answer(ans)
            texts, lps = [], []
            for _ in range(N_SAMPLES):
                t, lp = gen_with_logprob(q, tmp)
                texts.append(t); lps.append(lp)

            pred = extract_answer(texts[0])
            R1 = reward_r1(pred, gt); R2 = reward_r2(texts[0])
            R3 = reward_r3(texts, gt); R4 = reward_r4(texts[0])
            raw = reward_fn(R1, R2, R3, R4)
            rewards_ep.append(float(raw))

            if len(rewards_ep) >= 2:
                mu  = sum(rewards_ep)/len(rewards_ep)
                sig = (sum((r-mu)**2 for r in rewards_ep)/len(rewards_ep))**0.5
                rn  = (float(raw)-mu)/(sig+1e-8)
            else:
                rn = float(raw)

            loss = -rn * lps[0]
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(tmp.parameters(), 1.0)
            opt.step()

            del texts, lps, loss
            torch.cuda.empty_cache(); gc.collect()

        avg = sum(rewards_ep)/len(rewards_ep)
        print(f"  Epoch {epoch+1}/{EPOCHS}  avg_reward={avg:.3f}")

    return tmp

### 6.3 Train Ablations

In [ ]:
full_model  = train_ablation(reward_full,   "Full  R1+R2+R3-R4")
no_r2_model = train_ablation(reward_no_r2,  "No-R2 (−step validity)")
no_r3_model = train_ablation(reward_no_r3,  "No-R3 (−consistency)")
no_r4_model = train_ablation(reward_no_r4,  "No-R4 (−verbosity penalty)")


========================================  Full  R1+R2+R3-R4  ========================================


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  Epoch 1/2  avg_reward=-0.016
  Epoch 2/2  avg_reward=-0.016

========================================  No-R2 (−step validity)  ========================================


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  Epoch 1/2  avg_reward=-0.025
  Epoch 2/2  avg_reward=-0.023

========================================  No-R3 (−consistency)  ========================================


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  Epoch 1/2  avg_reward=-0.018
  Epoch 2/2  avg_reward=-0.020

========================================  No-R4 (−verbosity penalty)  ========================================


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  Epoch 1/2  avg_reward=0.000
  Epoch 2/2  avg_reward=0.000


### 6.4 Evaluate All Models

In [ ]:
print("Evaluating all models...\n")
baseline_acc,  baseline_cons,  baseline_step  = evaluate_model(baseline_model, eval_pool, label="Baseline")
full_acc,      full_cons,      full_step      = evaluate_model(full_model,      eval_pool, label="Full")
no_r2_acc,     no_r2_cons,     no_r2_step     = evaluate_model(no_r2_model,     eval_pool, label="No-R2")
no_r3_acc,     no_r3_cons,     no_r3_step     = evaluate_model(no_r3_model,     eval_pool, label="No-R3")
no_r4_acc,     no_r4_cons,     no_r4_step     = evaluate_model(no_r4_model,     eval_pool, label="No-R4")

Evaluating all models...

  [Baseline] 5/15
  [Baseline] 10/15
  [Baseline] 15/15
  [Full] 5/15
  [Full] 10/15
  [Full] 15/15
  [No-R2] 5/15
  [No-R2] 10/15
  [No-R2] 15/15
  [No-R3] 5/15
  [No-R3] 10/15
  [No-R3] 15/15
  [No-R4] 5/15
  [No-R4] 10/15
  [No-R4] 15/15


### 6.5 Final Results Table

In [ ]:
W = 14
print("\nFINAL RESULTS")
print("="*62)
print(f"{'Model':<{W}} | {'Accuracy':>8} | {'Consistency':>11} | {'Step Score':>10}")
print("="*62)
for name, acc, cons, step in [
    ("Baseline",  baseline_acc,  baseline_cons,  baseline_step),
    ("Full",      full_acc,      full_cons,      full_step),
    ("No R2",     no_r2_acc,     no_r2_cons,     no_r2_step),
    ("No R3",     no_r3_acc,     no_r3_cons,     no_r3_step),
    ("No R4",     no_r4_acc,     no_r4_cons,     no_r4_step),
]:
    print(f"{name:<{W}} | {acc:>8.3f} | {cons:>11.3f} | {step:>10.3f}")
print("="*62)
print("\nPrimary result : Consistency  (target +10–20% vs Baseline)")
print("Secondary      : Accuracy     (target +2–5%)")
print("Supporting     : Step Score   (target −8–12% error rate)")


FINAL RESULTS
Model          | Accuracy | Consistency | Step Score
Baseline       |    0.000 |       0.600 |      0.244
Full           |    0.000 |       0.667 |      0.302
No R2          |    0.000 |       0.800 |      0.304
No R3          |    0.000 |       0.733 |      0.136
No R4          |    0.000 |       0.867 |      0.230

Primary result : Consistency  (target +10–20% vs Baseline)
Secondary      : Accuracy     (target +2–5%)
Supporting     : Step Score   (target −8–12% error rate)
